# 📉 Cuaderno 03 — Mínimos cuadrados

**Métodos Numéricos · Unidad 4: Ajuste de curvas e interpolación**

En este cuaderno ajustarás curvas a datos experimentales con ruido: minimizarás la suma de residuos al cuadrado, compararás regresiones lineal y cuadrática y evaluarás sus errores.

In [ ]:
#@title ⚙️ Entorno — ejecuta esta celda PRIMERO
# ---------------------------------------------------------------
# En tu computadora usa los archivos locales, sin repositorio.
# En Google Colab pide UNA vez el enlace de tu fork, lo guarda en tu
# Drive y a partir de ahí lo reutiliza solo.
# ---------------------------------------------------------------
import json
import os
import subprocess
import sys

CUADERNO = 'u4_03_minimos_cuadrados'
TAREA_ID = 'U4_03_Minimos_Cuadrados'
REPO_NOMBRE = "U4_Repo"
GUARDADO = os.path.join("CursoMN", "u4_repo.json")   # relativo a MyDrive

try:
    import google.colab  # noqa: F401
    ES_COLAB = True
except ImportError:
    ES_COLAB = False


def _raiz_local():
    """Sube desde la carpeta actual hasta encontrar lib/robomat.py."""
    candidatos = []
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        candidatos.append(os.path.dirname(os.path.abspath(nb)))
    candidatos.append(os.getcwd())
    for base in candidatos:
        b = base
        for _ in range(6):
            if os.path.exists(os.path.join(b, "lib", "robomat.py")):
                return b
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


def _normaliza(enlace):
    enlace = (enlace or "").strip().rstrip("/")
    if enlace.endswith(".git"):
        enlace = enlace[:-4]
    if not enlace.startswith("https://github.com/"):
        raise ValueError("Necesito un enlace https://github.com/USUARIO/"
                         + REPO_NOMBRE + " (tu fork).")
    return enlace

# Repositorio del curso (upstream) y carpetas que el alumno nunca edita:
# se refrescan en cada ejecución para que el material y el motor nunca estén
# atrasados, aunque el fork se haya quedado viejo.
CURSO = "https://github.com/ITH-MGL-MN/" + REPO_NOMBRE
MATERIAL = ("lib", "grader", "config", "docs", "ejercicios", "tools")


def _git(*args):
    """Corre git dentro del clon. Devuelve (funcionó, salida)."""
    r = subprocess.run(["git", "-C", REPO] + list(args),
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()


def actualizar():
    """Trae tus commits de GitHub y refresca el material del curso.

    El clon de Drive es desechable: tus respuestas viven en el cuaderno.
    Primero se actualiza el fork (lo que editaste en GitHub) y después se
    copia el material del curso desde `upstream/main`. Si tienes cambios
    locales en el clon, no se tocan: se avisa y se sigue.
    """
    ok, _ = _git("rev-parse", "--is-inside-work-tree")
    if not ok:
        print("⚠️ La carpeta no es un repositorio git: uso los archivos tal cual.")
        return
    _git("remote", "add", "upstream", CURSO + ".git")
    ok, salida = _git("fetch", "--all", "--prune", "--quiet")
    if not ok:
        print("⚠️ No pude conectar con GitHub; sigo con lo que hay en Drive.")
        return

    _git("checkout", "--", *MATERIAL)          # limpia el refresco anterior
    _, rama = _git("rev-parse", "--abbrev-ref", "HEAD")
    rama = rama or "main"
    ok, salida = _git("merge", "--ff-only", "origin/" + rama)
    if ok:
        print("Fork        : al día con tus commits de GitHub")
    else:
        print("⚠️ Tu fork no avanza en línea recta (tienes commits propios")
        print("   o cambios locales): lo dejo como está y solo refresco el")
        print("   material del curso.")

    ok, salida = _git("checkout", "upstream/main", "--", *MATERIAL)
    if ok:
        _git("reset", "--quiet")
        _, sha = _git("rev-parse", "--short", "upstream/main")
        print("Material    : actualizado al curso", sha)
    else:
        print("⚠️ No pude refrescar el material del curso:", salida.splitlines()[:1])


if not ES_COLAB:
    REPO = _raiz_local()
    if REPO is None:
        raise RuntimeError(
            "No encontré la raíz del repositorio (lib/robomat.py).\n"
            "Abre el cuaderno dentro de U4_Repo o ejecútalo desde esa carpeta.")
    print("Modo        : local (sin repositorio)")
    print("Repositorio :", REPO)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    ruta_json = os.path.join(DRIVE, GUARDADO)

    guardado = {}
    if os.path.exists(ruta_json):
        try:
            with open(ruta_json, encoding="utf-8") as f:
                guardado = json.load(f)
        except (OSError, json.JSONDecodeError) as exc:
            raise RuntimeError(
                f"No se pudo leer la configuracion de Drive: {ruta_json}"
            ) from exc

    enlace = (guardado.get("repo") or "").strip()
    if enlace:
        print("✅ Repositorio guardado:", enlace)
        print("   (si cambió, edita la caja de abajo y pulsa el botón)")
    else:
        try:
            enlace = input("Pega el enlace de TU fork de U4_Repo: ")
        except Exception:
            enlace = ""
        enlace = (enlace or "").strip()
        if not enlace:
            raise RuntimeError(
                "Sin repositorio no puedo continuar en Colab.\n"
                "Haz fork del repositorio del curso y pega su enlace.")

    enlace = _normaliza(enlace)
    REPO = os.path.join(DRIVE, "CursoMN", REPO_NOMBRE)

    if not os.path.exists(REPO):
        print("Clonando tu repositorio…")
        subprocess.run(["git", "clone", enlace + ".git", REPO], check=True)
    actualizar()

    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump({"repo": enlace}, f, indent=1)
    print("Repositorio :", REPO)
    print("Actualizar  : vuelve a ejecutar esta celda cuando subas cambios a GitHub")

# Instala dependencias del repositorio solo si falta alguna.
import importlib.util

_modulos_requeridos = ("numpy", "scipy", "sympy", "matplotlib", "yaml", "requests")
_faltantes = [nombre for nombre in _modulos_requeridos
              if importlib.util.find_spec(nombre) is None]
if _faltantes:
    requisitos = os.path.join(REPO, "requirements.txt")
    if not os.path.isfile(requisitos):
        raise FileNotFoundError("Falta requirements.txt en el repositorio.")
    print("Instalando dependencias faltantes:", ", ".join(_faltantes))
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", requisitos],
                   check=True)

# Indica al motor empaquetado dónde está la configuración del repositorio.
os.environ["MN_REPO"] = REPO

# --- Motor de calificación -------------------------------------------
# Se usa el código fuente si está (rama `test`, para depurar); si no, el blob
# empaquetado de grader/, que es lo que recibe el alumno.
_ruta_modulos = os.path.join(REPO, "lib", "mn_motor")
_ruta_blob = os.path.join(REPO, "grader", CUADERNO + ".txt")

_lib = os.path.join(REPO, "lib")
if _lib not in sys.path:
    sys.path.insert(0, _lib)          # ajuste_curvas y, si está, mn_motor

if os.path.isfile(os.path.join(_ruta_modulos, "__init__.py")):
    import mn_motor as _motor
    globals().update({n: getattr(_motor, n) for n in _motor.__all__})
    print("Motor       : módulos de lib/mn_motor (modo profesor)")
else:
    import base64
    import zlib

    if not os.path.exists(_ruta_blob):
        raise FileNotFoundError(
            "No encuentro el motor: falta grader/" + CUADERNO + ".txt\n"
            "Clona el repositorio completo (incluye la carpeta grader/).")
    with open(_ruta_blob, encoding="utf-8") as f:
        codigo = zlib.decompress(base64.b64decode(f.read().strip())).decode("utf-8")
    _MOTOR = {"__name__": "mn_motor_blob", "__file__": "<blob>"}
    exec(compile(codigo, "<blob>", "exec"), _MOTOR)
    globals().update({n: _MOTOR[n] for n in _MOTOR["__all__"]})
    print("Motor       : blob verificado de grader/")

usar_cuaderno(CUADERNO, TAREA_ID)

# --- Preludio estilo MATLAB y herramientas de gráficas ---------------
with open(os.path.join(REPO, "lib", "robomat.py"), encoding="utf-8") as f:
    exec(f.read(), globals())

if REPO not in sys.path:
    sys.path.insert(0, REPO)          # para importar ejercicios/
sys.path.append(os.path.join(REPO, "tools"))
import importlib
import visualizar
importlib.reload(visualizar)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110
print("Preludio    : lib/robomat.py · tools/visualizar.py")
print("\nListo. Ejecuta la celda de identificación.")

In [ ]:
#@title 👤 Identificación y semilla reproducible de preguntas
import os

if ES_COLAB:
    import requests
    from google.colab import auth
    import google.auth
    import google.auth.transport.requests

    auth.authenticate_user()
    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    respuesta = requests.get(
        "https://www.googleapis.com/drive/v3/about?fields=user",
        headers={"Authorization": "Bearer " + creds.token},
        timeout=20,
    )
    respuesta.raise_for_status()
    cuenta = respuesta.json().get("user", {})
    email = cuenta.get("emailAddress", "").strip()
    nombre = cuenta.get("displayName", "")
else:
    email = os.environ.get("MN_EMAIL", "").strip()
    if not email:
        email = input("Escribe tu correo institucional @tecnm.mx: ").strip()
    nombre = ""

if not (email and "@" in email and email.lower().endswith(".tecnm.mx")):
    raise ValueError("Usa tu correo institucional terminado en .tecnm.mx.")

MI_TAREA = generar_tarea(email, nombre)
semilla = MI_TAREA["semilla"]
print("Alumno:", email)
print("NC / semilla reproducible:", extraer_nc(email), "/", semilla)
print("Cuaderno:", MI_TAREA["cuaderno"], "·", len(MI_TAREA["preguntas"]), "preguntas")


---

## Método 1 · Suma de residuos al cuadrado

### Puente con optimización (capítulo 4)

En optimización se busca el mínimo o máximo de una función. Aquí la función objetivo depende de los coeficientes del modelo: se eligen para minimizar $S_r$. La interpolación parabólica del capítulo 4 ilustra cómo aproximar un extremo; en mínimos cuadrados el extremo buscado es un **mínimo** respecto a los parámetros del ajuste.

![Capítulo 4, figura 13.6: interpolación parabólica para localizar un extremo](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/main/docs/figuras/libro/fig13-6_parabolica.jpg)

### Criterio de mínimos cuadrados (capítulo 5)

Para cada dato, el residuo es la diferencia vertical entre la medición y el modelo:

$$e_i=y_i-y_{i,\text{modelo}}.$$

El criterio es minimizar

$$S_r=\sum_{i=1}^{n}e_i^2=\sum_{i=1}^{n}\left(y_i-a_0-a_1x_i-\cdots-a_mx_i^m\right)^2. \tag{17.3, 17.18}$$

La suma con signo puede cancelarse aunque los errores sean grandes; la suma de cuadrados evita esa cancelación y define un criterio único para los coeficientes. A diferencia de la interpolación, el ajuste no exige pasar por cada punto ruidoso.

![Capítulo 5, figura 17.2: por qué otros criterios de ajuste pueden ser inadecuados](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/main/docs/figuras/libro/fig17-2_criterios.jpg)


In [ ]:
#@title ✏️ Actividad a mano · Suma de residuos al cuadrado
EJ_1 = mano_enunciado("SUMA_RESIDUOS")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Suma de residuos al cuadrado
# Organiza yi, y estimada, residuo y residuo^2 en una tabla; reporta Sr con 4 decimales.
mano_comprueba("SUMA_RESIDUOS", [0.0])

### 💻 Programa la función

Edita `ejercicios/minimos_cuadrados/suma_residuos_cuadrados.py` en GitHub, edítalo ahí y confirma el cambio. Después vuelve a ejecutar la celda de entorno (trae tu commit) y la celda de importación.

El residuo es $y_i-\hat y_i$ y los coeficientes se reciben en orden creciente: `[a0, a1, ...]`.


In [ ]:
#@title 💻 Tu función · Suma de residuos al cuadrado
from ejercicios.minimos_cuadrados.suma_residuos_cuadrados import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("SUMA_RESIDUOS")

In [ ]:
#@title 📊 Laboratorio · Suma de residuos al cuadrado
visualizar.laboratorio("SUMA_RESIDUOS", EJ_1)

In [ ]:
#@title 🧪 Pregunta 1 · Suma de residuos al cuadrado
# Cuando tu función ya funcione, contéstala aquí.
pregunta(1)

---

## Método 2 · Regresión lineal

El modelo para datos con dispersión es

$$y_i=a_0+a_1x_i+e_i.$$

Al minimizar $S_r$ respecto a $a_0$ y $a_1$, las condiciones $\partial S_r/\partial a_0=0$ y $\partial S_r/\partial a_1=0$ producen las ecuaciones normales:

$$n a_0+\left(\sum x_i\right)a_1=\sum y_i, \tag{17.4}$$

$$\left(\sum x_i\right)a_0+\left(\sum x_i^2\right)a_1=\sum x_i y_i. \tag{17.5}$$

Su solución es

$$a_1=\frac{n\sum x_i y_i-\left(\sum x_i\right)\left(\sum y_i\right)}{n\sum x_i^2-\left(\sum x_i\right)^2},\qquad a_0=\bar y-a_1\bar x. \tag{17.6–17.7}$$

**Matriz en Python.** Con las sumas de tu tabla, construye `A = M[[n, Sx], [Sx, Sx2]]` y `b = M[[Sy], [Sxy]]`; para mostrar el sistema aumentado usa `M[[A, b]]`. En papel escribe la misma matriz aumentada con notación convencional.

![Capítulo 5, figura 17.4: dispersión alrededor de la media y de la recta ajustada](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/main/docs/figuras/libro/fig17-4_dispersion.jpg)


In [ ]:
#@title ✏️ Actividad a mano · Regresión lineal
EJ_2 = mano_enunciado("REG_LINEAL")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Regresión lineal
# Completa las sumas y resuelve las ecuaciones normales. En papel usa matriz aumentada;
# en código, A=M[[n,Sx],[Sx,Sx2]], b=M[[Sy],[Sxy]], M[[A,b]] (robomat).
# Reporta los coeficientes con 4 decimales.
mano_comprueba("REG_LINEAL", [0.0, 0.0, 0.0])

### 💻 Programa la función

Edita `ejercicios/minimos_cuadrados/regresion_lineal.py` en GitHub, edítalo ahí y confirma el cambio. Después vuelve a ejecutar la celda de entorno (trae tu commit) y la celda de importación.

La función completa debe llamar a `suma_residuos_cuadrados` para cuantificar el ajuste; no dupliques esa cuenta.


In [ ]:
#@title 💻 Tu función · Regresión lineal
from ejercicios.minimos_cuadrados.regresion_lineal import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("REG_LINEAL")

In [ ]:
#@title 📊 Laboratorio · Regresión lineal
visualizar.laboratorio("REG_LINEAL", EJ_2)

In [ ]:
#@title 🧪 Pregunta 2 · Regresión lineal
# Cuando tu función ya funcione, contéstala aquí.
pregunta(2)

---

## Método 3 · Regresión cuadrática

Si la tendencia presenta curvatura, el modelo puede ser

$$y_i=a_0+a_1x_i+a_2x_i^2+e_i,$$

con

$$S_r=\sum_{i=1}^{n}(y_i-a_0-a_1x_i-a_2x_i^2)^2. \tag{17.18}$$

Al igualar a cero las derivadas respecto a los tres coeficientes se obtienen las ecuaciones normales:

$$\begin{bmatrix}
n&\sum x_i&\sum x_i^2\\
\sum x_i&\sum x_i^2&\sum x_i^3\\
\sum x_i^2&\sum x_i^3&\sum x_i^4
\end{bmatrix}
\begin{bmatrix}a_0\\a_1\\a_2\end{bmatrix}=
\begin{bmatrix}\sum y_i\\\sum x_i y_i\\\sum x_i^2y_i\end{bmatrix}. \tag{17.19}$$

En Python puedes construir `A = M[[n, Sx, Sx2], [Sx, Sx2, Sx3], [Sx2, Sx3, Sx4]]`, `b = M[[Sy], [Sxy], [Sx2y]]`, y la aumentada `M[[A, b]]`. La solución es un problema de álgebra lineal de la Unidad 3.

![Capítulo 5, figura 17.11: ajuste cuadrático a datos experimentales](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/main/docs/figuras/libro/fig17-11_regresion_cuadratica.jpg)

Agregar términos puede reducir $S_r$, pero no justifica por sí solo un grado mayor: revisa los residuales, el error estándar y la complejidad del modelo.


In [ ]:
#@title ✏️ Actividad a mano · Regresión cuadrática
EJ_3 = mano_enunciado("REG_CUADRATICA")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Regresión cuadrática
# Organiza las sumas; en Python construye A y b con M[[fila1], ...] y muestra M[[A,b]].
# Usa los constructores MATLAB-like de robomat, no np.array/np.zeros.
# Reporta los coeficientes y Sr con 4 decimales.
mano_comprueba("REG_CUADRATICA", [0.0, 0.0, 0.0, 0.0])

### 💻 Programa la función

Edita `ejercicios/minimos_cuadrados/regresion_cuadratica.py` en GitHub, edítalo ahí y confirma el cambio. Después vuelve a ejecutar la celda de entorno (trae tu commit) y la celda de importación.

Resuelve las ecuaciones normales y después llama a `suma_residuos_cuadrados` para obtener $S_r$. Los coeficientes regresan en orden `[a0, a1, a2]`.


In [ ]:
#@title 💻 Tu función · Regresión cuadrática
from ejercicios.minimos_cuadrados.regresion_cuadratica import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("REG_CUADRATICA")

In [ ]:
#@title 📊 Laboratorio · Regresión cuadrática
visualizar.laboratorio("REG_CUADRATICA", EJ_3)

In [ ]:
#@title 🧪 Pregunta 3 · Regresión cuadrática
# Cuando tu función ya funcione, contéstala aquí.
pregunta(3)

---

## Método 4 · Error estándar de la estimación

Para un polinomio de grado $m$,

$$s_{y/x}=\sqrt{\frac{S_r}{n-(m+1)}}. \tag{17.9, 17.20}$$

Se descuentan $m+1$ grados de libertad porque esos coeficientes se estimaron con los mismos datos. La medida se expresa en las unidades de $y$ y resume la dispersión de los datos alrededor del modelo; si $n\le m+1$, no quedan grados de libertad para estimarla.

![La figura 17.4 compara la dispersión alrededor de la media con la dispersión alrededor de la recta ajustada](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/main/docs/figuras/libro/fig17-4_dispersion.jpg)


In [ ]:
#@title ✏️ Actividad a mano · Error estándar de la estimación
EJ_4 = mano_enunciado("ERROR_ESTANDAR")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Error estándar de la estimación
# Aplica la fórmula de sy/x usando Sr y los grados de libertad; reporta 4 decimales.
mano_comprueba("ERROR_ESTANDAR", [0.0])

### 💻 Programa la función

Edita `ejercicios/minimos_cuadrados/error_estandar.py` en GitHub, edítalo ahí y confirma el cambio. Después vuelve a ejecutar la celda de entorno (trae tu commit) y la celda de importación.

Usa el grado $m$ del modelo, no el número de coeficientes; valida que $n-(m+1)>0$.


In [ ]:
#@title 💻 Tu función · Error estándar de la estimación
from ejercicios.minimos_cuadrados.error_estandar import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("ERROR_ESTANDAR")

In [ ]:
#@title 📊 Laboratorio · Error estándar de la estimación
visualizar.laboratorio("ERROR_ESTANDAR", EJ_4)

In [ ]:
#@title 🧪 Pregunta 4 · Error estándar de la estimación
# Cuando tu función ya funcione, contéstala aquí.
pregunta(4)

---

## Método 5 · Coeficiente de determinación

La variabilidad inicial de los datos respecto a su media es

$$S_t=\sum_{i=1}^{n}(y_i-\bar y)^2,$$

y la variabilidad que queda sin explicar por el modelo es $S_r$. El coeficiente de determinación es

$$r^2=\frac{S_t-S_r}{S_t}. \tag{17.10}$$

Un $r^2=0.95$ indica que el modelo explica alrededor del 95% de la variabilidad de $y$ respecto a su media. No demuestra que la forma del modelo sea adecuada: inspecciona la gráfica y los residuales. Si $S_t=0$, los datos no tienen variación y esta razón no está definida.


In [ ]:
#@title ✏️ Actividad a mano · Coeficiente de determinación
EJ_5 = mano_enunciado("COEF_DETERMINACION")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Coeficiente de determinación
# Usa la fórmula de r^2 y muestra las sumas sustituidas; reporta 4 decimales.
mano_comprueba("COEF_DETERMINACION", [0.0])

### 💻 Programa la función

Edita `ejercicios/minimos_cuadrados/coeficiente_determinacion.py` en GitHub, edítalo ahí y confirma el cambio. Después vuelve a ejecutar la celda de entorno (trae tu commit) y la celda de importación.

Usa $S_t$ alrededor de la media y $S_r$ alrededor del modelo; no confundas ambas sumas.


In [ ]:
#@title 💻 Tu función · Coeficiente de determinación
from ejercicios.minimos_cuadrados.coeficiente_determinacion import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("COEF_DETERMINACION")

In [ ]:
#@title 📊 Laboratorio · Coeficiente de determinación
visualizar.laboratorio("COEF_DETERMINACION", EJ_5)

In [ ]:
#@title 🧪 Pregunta 5 · Coeficiente de determinación
# Cuando tu función ya funcione, contéstala aquí.
pregunta(5)

In [ ]:
#@title 📊 Laboratorio integrador · comparar modelos con datos del libro
from IPython.display import Markdown, display
from ajuste_curvas import minimos_cuadrados

# Ejemplo 17.5 y tabla 17.4 del capítulo 5.
x_libro = [0, 1, 2, 3, 4, 5]
y_libro = [2.1, 7.7, 13.6, 27.2, 40.9, 61.1]
resultados_libro = {
    grado: minimos_cuadrados(x_libro, y_libro, grado)
    for grado in (1, 2)
}

filas = [
    "| Grado | Coeficientes `[a0, a1, ...]` | $S_r$ | $s_{y/x}$ | $r^2$ |",
    "|---:|:--|--:|--:|--:|",
]
for grado, resultado in resultados_libro.items():
    coef = ", ".join(f"{float(a):.5f}" for a in resultado["coef"])
    filas.append(
        f"| {grado} | `{coef}` | {resultado['Sr']:.5f} | "
        f"{resultado['syx']:.5f} | {resultado['r2']:.6f} |"
    )
display(Markdown("## Caso del libro · Ejemplo 17.5\n\n" + "\n".join(filas)))

x_malla = [x_libro[0] + (x_libro[-1] - x_libro[0]) * i / 200
           for i in range(201)]
fig, (ax_ajuste, ax_res) = plt.subplots(1, 2, figsize=(11, 4.2))
ax_ajuste.scatter(x_libro, y_libro, color="black", label="datos de la tabla 17.4")
for grado, resultado in resultados_libro.items():
    coef = [float(a) for a in resultado["coef"]]
    y_malla = [sum(a * x ** k for k, a in enumerate(coef)) for x in x_malla]
    y_pred = [sum(a * x ** k for k, a in enumerate(coef)) for x in x_libro]
    residuales = [yi - yp for yi, yp in zip(y_libro, y_pred)]
    ax_ajuste.plot(x_malla, y_malla, label=f"grado {grado}")
    ax_res.scatter(x_libro, residuales, label=f"grado {grado}")
ax_ajuste.set_title("Ajuste de los datos")
ax_ajuste.set_xlabel("x")
ax_ajuste.set_ylabel("y")
ax_ajuste.grid(True, alpha=0.3)
ax_ajuste.legend()
ax_res.axhline(0, color="black", linewidth=0.8)
ax_res.set_title("Residuales: y medida − y modelo")
ax_res.set_xlabel("x")
ax_res.set_ylabel("residuo")
ax_res.grid(True, alpha=0.3)
ax_res.legend()
plt.tight_layout()
plt.show()

print("Compara la estructura de los residuales y s_y/x; no elijas el grado solo por r².")


In [ ]:
#@title ✅ 1 · Verificación de los ejercicios a mano
mano_comprueba_todos(["SUMA_RESIDUOS", "REG_LINEAL", "REG_CUADRATICA", "ERROR_ESTANDAR", "COEF_DETERMINACION"])

In [ ]:
#@title 📝 2 · Verificación de todas las preguntas (no envía nada)
puntos, maximo = calificar()

In [ ]:
#@title 📤 3 · Envío de las respuestas
enviar(email)

In [ ]:
#@title 🔎 4 · Verificación de lo que ya está enviado (no envía nada)
consultar_calificacion()